# Create one managed and one external Delta table over the same data on a Premium trial, then prove DROP TABLE removes files for one and leaves them for the other

A migration team is moving an existing S3 (or ADLS, or GCS) resident dataset into Unity Catalog and security has been very clear: the bytes do not leave the team's bucket. You have one session to spin up the Premium trial, wire the storage credential and external location, write the same dataset twice (once registered as managed, once as external), and prove DROP TABLE removes the underlying files for one and leaves them for the other. The exam tests the distinction directly; the migration team needs the muscle memory.

The hands-on work:

- Create a cloud-side IAM role (AWS), managed identity (Azure), or service account (GCP) that grants Databricks read/write on a bucket prefix you control, then create a Unity Catalog storage credential and external location pointing at it.
- Stage 100 rows of synthetic Parquet data twice under the external location prefix: once at `.../managed-source/` and once at `.../external-source/`.
- Create `products_managed` from the managed-source path WITHOUT a `LOCATION` clause (Unity Catalog copies the bytes into the workspace storage location and owns them).
- Create `products_external` over the external-source path WITH a `LOCATION` clause (Unity Catalog leaves the bytes in your bucket and only owns the metastore registration).
- DROP each table; LIST the bucket prefix after each DROP; confirm managed files are gone and external files are intact.

**Time.** Plan for about 65 minutes hands-on if your cloud-side IAM is already comfortable; first-time Databricks-to-cloud trust setup adds 15-20 minutes. Budget up to 120 minutes for the session window.

**Cost.** This is the FIRST Databricks lab where the dollar number is not zero. The Premium 14-day trial itself is free, but the underlying cloud workspace runs serverless or classic compute against your cloud quota. Expect $0.05 to $0.50 per session in cloud compute and a few cents in storage. The line item that bites is forgetting to tear down the trial workspace before day 14, at which point the trial converts to paid and you start spending real money. The cleanup cell prints the date you need to act on.

This lab maps to Databricks DE Associate Domain 7 (Governance and Security, ~11 percent provisional).

**Premium trial reminder.** A Databricks Premium trial is a single 14-day window per cloud account. Run Labs 10, 11, and 12 consecutively inside one trial. Set a billing alert at 10 USD on your cloud provider before activating the trial.

**Rename callout.** If your other prep material says "Hive metastore tables" in the context of managed vs external, that vocabulary is outdated. The current exam is Unity Catalog only. The managed-vs-external distinction is structurally similar to Hive's but the storage-credential plus external-location pattern is Unity Catalog-specific.

**Local prerequisites.** This lab runs from Colab against your Databricks Premium trial workspace via the Statement Execution API. You will paste three getpass prompts: workspace URL, personal access token, and the bucket prefix you want the external location to point at (something like `s3://student-bucket/labrun-external/` or `abfss://container@account.dfs.core.windows.net/labrun-external/` or `gs://student-bucket/labrun-external/`).

In [ ]:
# NBVAL_SKIP
# Install labrun-checks and the Databricks SDK. Pinned versions per
# LAB-CREATION-BLUEPRINT.md build rules.

!pip install --quiet labrun-checks==0.6.0 databricks-sdk==0.40.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. CATALOG defaults to 'workspace' on a fresh
# Premium trial; if you created a custom catalog, update CATALOG below.

import atexit
import getpass
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "managed-vs-external-tables-on-unity-catalog"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID

CATALOG = "workspace"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_managed_vs_external"
SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"

STORAGE_CREDENTIAL_NAME = "labrun-storage-credential"
EXTERNAL_LOCATION_NAME = "labrun-external-location"

MANAGED_TABLE_FQN = f"{SCHEMA_FQN}.products_managed"
EXTERNAL_TABLE_FQN = f"{SCHEMA_FQN}.products_external"

# Filled by the student in Task 1 (cloud-side principal id) and Task 3/4
# (captured pre-drop locations).
cloud_principal_id = None
external_location_url = None
managed_source_path = None
external_source_path = None
managed_location_pre_drop = None
external_files_pre_drop = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate Databricks credentials, capture the bucket
# prefix the student controls, resolve the SQL warehouse, expose run_sql().

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Databricks workspace URL: ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT): ").strip()
bucket_prefix = getpass.getpass(
    "Bucket prefix for the external location (e.g., s3://your-bucket/labrun-external/): "
).strip()

if not databricks_host or not databricks_token or not bucket_prefix:
    print("Workspace URL, PAT, and bucket prefix are all required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"
databricks_host = databricks_host.rstrip("/")
if not bucket_prefix.endswith("/"):
    bucket_prefix = bucket_prefix + "/"

EXTERNAL_LOCATION_URL = bucket_prefix
MANAGED_SOURCE_PATH = bucket_prefix + "managed-source/"
EXTERNAL_SOURCE_PATH = bucket_prefix + "external-source/"
external_location_url = EXTERNAL_LOCATION_URL
managed_source_path = MANAGED_SOURCE_PATH
external_source_path = EXTERNAL_SOURCE_PATH

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired:")
    print(f"  {exc}")
    raise SystemExit(1)
CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found. Create one in the trial workspace and re-run.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")
print(f"External location URL:  {EXTERNAL_LOCATION_URL}")
print(f"Managed source path:    {MANAGED_SOURCE_PATH}")
print(f"External source path:   {EXTERNAL_SOURCE_PATH}")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
}


def run_sql(statement, warehouse_id=None, wait_seconds=240, ignore_errors=False):
    wid = warehouse_id or WAREHOUSE_ID
    try:
        resp = w.statement_execution.execute_statement(
            warehouse_id=wid, statement=statement, wait_timeout="50s",
        )
        statement_id = resp.statement_id
        deadline = time.time() + wait_seconds
        while True:
            state = resp.status.state if resp.status else None
            if state == StatementState.SUCCEEDED:
                break
            if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
                err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
                if ignore_errors:
                    return {"columns": [], "rows": [], "statement_id": statement_id, "error": err}
                raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement[:200]}")
            if time.time() > deadline:
                raise TimeoutError(f"SQL did not finish in {wait_seconds}s. Last state: {state}.")
            time.sleep(2)
            resp = w.statement_execution.get_statement(statement_id)
        columns = []
        if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
            columns = [c.name for c in resp.manifest.schema.columns]
        rows = []
        if resp.result and resp.result.data_array:
            rows = list(resp.result.data_array)
        return {"columns": columns, "rows": rows, "statement_id": statement_id, "error": None}
    except Exception as exc:
        if ignore_errors:
            return {"columns": [], "rows": [], "statement_id": None, "error": str(exc)}
        raise


def list_path(uri):
    """LIST a cloud path via SQL. Returns rows of (path, name, size, modificationTime)
    or empty if the path does not exist. Errors propagate so the caller can
    distinguish 'no files' from 'no permission'."""
    return run_sql(f"LIST '{uri}'", ignore_errors=True)


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, adapter, atexit, orphan scan. The two tables, the
# schema, the external location, and the storage credential live in
# Databricks. The cloud-side IAM principal and the bucket prefix live in
# the student's cloud account; the cleanup cell prints reminders for them
# and lists the bucket so the student can confirm the bytes are gone.


CLEANUP_MANIFEST = [
    CleanupResource(
        type="uc_external_table",
        id=EXTERNAL_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {EXTERNAL_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=MANAGED_TABLE_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {MANAGED_TABLE_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {SCHEMA_FQN} CASCADE\"",
    ),
    CleanupResource(
        type="uc_external_location",
        id=EXTERNAL_LOCATION_NAME,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP EXTERNAL LOCATION IF EXISTS {EXTERNAL_LOCATION_NAME}\"",
    ),
    CleanupResource(
        type="uc_storage_credential",
        id=STORAGE_CREDENTIAL_NAME,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP STORAGE CREDENTIAL IF EXISTS {STORAGE_CREDENTIAL_NAME}\"",
    ),
]


class DatabricksCleanupAdapter:
    def delete_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype in ("uc_managed_table", "uc_external_table"):
            run_sql(f"DROP TABLE IF EXISTS {rid}", ignore_errors=True)
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE", ignore_errors=True)
        elif rtype == "uc_external_location":
            run_sql(f"DROP EXTERNAL LOCATION IF EXISTS {rid}", ignore_errors=True)
        elif rtype == "uc_storage_credential":
            run_sql(f"DROP STORAGE CREDENTIAL IF EXISTS {rid}", ignore_errors=True)
        else:
            raise ValueError(f"Unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype, rid = resource.type, resource.id
        if rtype in ("uc_managed_table", "uc_external_table"):
            catalog, schema, table = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_schema":
            catalog, schema = rid.split(".")
            sql = (
                "SELECT 1 FROM system.information_schema.schemata "
                f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
            )
            return len(run_sql(sql)["rows"]) > 0
        if rtype == "uc_external_location":
            sql = (
                "SELECT 1 FROM system.information_schema.external_locations "
                f"WHERE external_location_name = '{rid}'"
            )
            res = run_sql(sql, ignore_errors=True)
            return len(res["rows"]) > 0
        if rtype == "uc_storage_credential":
            sql = (
                "SELECT 1 FROM system.information_schema.storage_credentials "
                f"WHERE storage_credential_name = '{rid}'"
            )
            res = run_sql(sql, ignore_errors=True)
            return len(res["rows"]) > 0
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"schema:{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"table:{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql, ignore_errors=True)
            except Exception:
                continue
            for row in (result.get("rows") or []):
                arns.append(fmt(row))
        return arns


CLEANUP_ADAPTER = DatabricksCleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


orphans = CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")
if orphans:
    print(f"BLOCKED: existing objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Create the cloud-side principal, the storage credential, and the external location

Three steps. The first happens in your cloud console; the next two happen as SQL on your Databricks SQL warehouse.

**Step A (cloud console).** Create a cloud-side principal with read/write/list/delete on the bucket prefix you provided in the setup cell:

- AWS: an IAM role with `s3:GetObject, s3:PutObject, s3:ListBucket, s3:DeleteObject` on `arn:aws:s3:::<your-bucket>/labrun-external/*` plus `s3:ListBucket` on `arn:aws:s3:::<your-bucket>` (with a `Condition` matching the `labrun-external/*` prefix). Trust the Databricks AWS account per the official Databricks "Set up Unity Catalog" docs.
- Azure: a managed identity (or service principal) with `Storage Blob Data Contributor` on the ADLS container.
- GCP: a service account with `storage.objects.{get,create,delete,list}` on the bucket.

Copy the principal's identifier:

- AWS: the IAM role ARN, e.g., `arn:aws:iam::123456789012:role/databricks-storage-credential-role`.
- Azure: the managed identity resource id, e.g., `/subscriptions/.../providers/Microsoft.ManagedIdentity/userAssignedIdentities/databricks-storage-credential`.
- GCP: the service account email, e.g., `databricks-storage@<project>.iam.gserviceaccount.com`.

Paste it into `cloud_principal_id` below.

**Step B (Databricks).** Create the storage credential bound to that principal:

```sql
-- AWS
CREATE STORAGE CREDENTIAL labrun-storage-credential
WITH AWS_IAM_ROLE 'arn:aws:iam::123456789012:role/databricks-storage-credential-role'
COMMENT 'labrun lab 10 storage credential';
```

(Substitute the equivalent `WITH AZURE_MANAGED_IDENTITY` or `WITH GCP_SERVICE_ACCOUNT_KEY` for Azure / GCP per the Databricks docs.)

**Step C (Databricks).** Create the external location pointing at your bucket prefix using the credential:

```sql
CREATE EXTERNAL LOCATION labrun-external-location
URL '<your bucket prefix>'
WITH (STORAGE CREDENTIAL labrun-storage-credential)
COMMENT 'labrun lab 10 external location';
```

Tag both the storage credential and the external location with the lab slug so the orphan scan sees them.

In [ ]:
# NBVAL_SKIP
# Task 1: Set cloud_principal_id, then create the storage credential and
# the external location with tags.

global cloud_principal_id

# YOUR CODE: Assign cloud_principal_id to your cloud-side principal id
# (IAM role ARN on AWS, managed identity resource id on Azure, service
# account email on GCP).
# cloud_principal_id = "arn:aws:iam::123456789012:role/databricks-storage-credential-role"

if not cloud_principal_id:
    print("cloud_principal_id is None. Set it to your cloud-side principal id.")
    raise SystemExit(1)

# YOUR CODE: Create the storage credential. Pick the right WITH clause for
# your cloud (AWS_IAM_ROLE, AZURE_MANAGED_IDENTITY, GCP_SERVICE_ACCOUNT_KEY).
# Example for AWS:
#   run_sql(f"CREATE STORAGE CREDENTIAL IF NOT EXISTS {STORAGE_CREDENTIAL_NAME} "
#           f"WITH AWS_IAM_ROLE '{cloud_principal_id}'")

# YOUR CODE: Tag the storage credential.
#   run_sql(f"ALTER STORAGE CREDENTIAL {STORAGE_CREDENTIAL_NAME} "
#           f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

# YOUR CODE: Create the external location.
#   run_sql(f"CREATE EXTERNAL LOCATION IF NOT EXISTS {EXTERNAL_LOCATION_NAME} "
#           f"URL '{EXTERNAL_LOCATION_URL}' "
#           f"WITH (STORAGE CREDENTIAL {STORAGE_CREDENTIAL_NAME})")

# YOUR CODE: Tag the external location.
#   run_sql(f"ALTER EXTERNAL LOCATION {EXTERNAL_LOCATION_NAME} "
#           f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

print(f"Cloud principal id:        {cloud_principal_id}")
print(f"Storage credential name:   {STORAGE_CREDENTIAL_NAME}")
print(f"External location name:    {EXTERNAL_LOCATION_NAME}")
print(f"External location URL:     {EXTERNAL_LOCATION_URL}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Storage credential and external location both exist on the
# trial metastore. The external location URL matches the bucket prefix the
# student provided. Both objects carry the lab tag.


def checkpoint_1(session):
    try:
        if not cloud_principal_id:
            return CheckpointResult(
                status="fail",
                error_reason="cloud_principal_id is not set. Assign your cloud-side principal id.",
            )

        sc_rows = run_sql(
            "SELECT storage_credential_name FROM system.information_schema.storage_credentials "
            f"WHERE storage_credential_name = '{STORAGE_CREDENTIAL_NAME}'",
            ignore_errors=True,
        )
        if sc_rows.get("error") and "PERMISSION_DENIED" in (sc_rows.get("error") or ""):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "system.information_schema.storage_credentials read denied. "
                    "Confirm you are signed in as a metastore admin on the trial."
                ),
            )
        if not sc_rows["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Storage credential {STORAGE_CREDENTIAL_NAME!r} not found. "
                    f"Run CREATE STORAGE CREDENTIAL with the right WITH clause "
                    f"for your cloud."
                ),
            )

        el_rows = run_sql(
            "SELECT external_location_name, url, storage_credential_name "
            "FROM system.information_schema.external_locations "
            f"WHERE external_location_name = '{EXTERNAL_LOCATION_NAME}'",
            ignore_errors=True,
        )
        if not el_rows["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"External location {EXTERNAL_LOCATION_NAME!r} not found. "
                    f"Run CREATE EXTERNAL LOCATION pointing at {EXTERNAL_LOCATION_URL!r}."
                ),
            )
        _, url, credential = el_rows["rows"][0]
        normalized_url = (url or "").rstrip("/") + "/"
        if normalized_url != EXTERNAL_LOCATION_URL:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"External location URL is {url!r}; expected {EXTERNAL_LOCATION_URL!r}."
                ),
            )
        if credential != STORAGE_CREDENTIAL_NAME:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"External location uses credential {credential!r}; expected "
                    f"{STORAGE_CREDENTIAL_NAME!r}."
                ),
            )

        # Tag presence on both objects (best-effort; some Databricks versions
        # surface tags via separate _tags views; tolerate missing tag tables).
        for obj_table, where_col in (
            ("storage_credential_tags", "storage_credential_name"),
            ("external_location_tags", "external_location_name"),
        ):
            obj_id = STORAGE_CREDENTIAL_NAME if "storage" in obj_table else EXTERNAL_LOCATION_NAME
            tag_sql = (
                f"SELECT tag_value FROM system.information_schema.{obj_table} "
                f"WHERE {where_col} = '{obj_id}' AND tag_name = '{LAB_TAG_KEY}'"
            )
            tag_res = run_sql(tag_sql, ignore_errors=True)
            if tag_res.get("error"):
                # Tag table not exposed on this trial; do not block the checkpoint.
                continue
            if not any(r[0] == LAB_TAG_VALUE for r in tag_res.get("rows", [])):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{obj_id} missing tag {LAB_TAG_KEY}={LAB_TAG_VALUE}. "
                        f"Run ALTER ... SET TAGS to add it."
                    ),
                )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is missing: cloud_principal_id, the storage credential row, the external location row, the URL, the credential reference, or the tag. Each is a separate SQL or Python line.

</details>

<details><summary>Hint 2 (direction)</summary>

Two CREATE statements (storage credential, external location) plus two ALTER statements (one tag each). The CREATE STORAGE CREDENTIAL syntax differs by cloud: `WITH AWS_IAM_ROLE`, `WITH AZURE_MANAGED_IDENTITY`, or `WITH GCP_SERVICE_ACCOUNT_KEY`. The CREATE EXTERNAL LOCATION URL must match the bucket prefix you provided in the setup cell exactly (with the trailing slash).

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
global cloud_principal_id
cloud_principal_id = "arn:aws:iam::123456789012:role/databricks-storage-credential-role"

# Storage credential. Pick the line that matches your cloud:
run_sql(
    f"CREATE STORAGE CREDENTIAL IF NOT EXISTS {STORAGE_CREDENTIAL_NAME} "
    f"WITH AWS_IAM_ROLE '{cloud_principal_id}'"
)
# Azure: WITH AZURE_MANAGED_IDENTITY 'subscriptions/.../userAssignedIdentities/...'
# GCP:   WITH GCP_SERVICE_ACCOUNT_KEY 'service-account-email@project.iam.gserviceaccount.com'

run_sql(
    f"ALTER STORAGE CREDENTIAL {STORAGE_CREDENTIAL_NAME} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)

run_sql(
    f"CREATE EXTERNAL LOCATION IF NOT EXISTS {EXTERNAL_LOCATION_NAME} "
    f"URL '{EXTERNAL_LOCATION_URL}' "
    f"WITH (STORAGE CREDENTIAL {STORAGE_CREDENTIAL_NAME})"
)
run_sql(
    f"ALTER EXTERNAL LOCATION {EXTERNAL_LOCATION_NAME} "
    f"SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')"
)
```

</details>

**Wallet check.** A few cents at most. Storage credentials and external locations are metadata; the cloud-side IAM principal is free. The serverless warehouse spent a few seconds running the four DDL statements. Cumulative session damage: under $0.05.

## Task 2: Stage two source datasets, then create one managed and one external table

The lab uses 100 rows of synthetic product data. You'll write the same dataset to two different paths under the external location prefix:

- `<bucket-prefix>managed-source/` (used to populate the MANAGED table)
- `<bucket-prefix>external-source/` (used as the LOCATION of the EXTERNAL table)

The simplest way to write Parquet from the SQL warehouse is to CREATE EXTERNAL TABLE temporarily at each path, INSERT 100 rows, then DROP the staging table (the bytes remain because external tables don't delete files on drop, which is exactly the lab's lesson).

Then create the actual lab tables:

- `products_managed`: `CREATE TABLE products_managed (...)` WITHOUT a `LOCATION` clause. Unity Catalog copies the Parquet bytes from the source path into the workspace storage location and owns them.
- `products_external`: `CREATE TABLE products_external (...) LOCATION '<external-source path>'`. Unity Catalog leaves the bytes at that path and only owns the metastore registration.

After Task 2, you have two tables that look identical to a SELECT query but live in completely different storage realities.

In [ ]:
# NBVAL_SKIP
# Task 2: Create the schema, stage two parquet datasets via temporary
# external staging tables, then create the actual managed and external
# lab tables.

# YOUR CODE: CREATE SCHEMA IF NOT EXISTS SCHEMA_FQN.
# YOUR CODE: ALTER SCHEMA SCHEMA_FQN SET TAGS.

# Stage Delta files at managed-source path via a temporary external staging
# table, INSERT 100 rows, then DROP the staging table (bytes remain).
# YOUR CODE:
#   run_sql(f"CREATE TABLE IF NOT EXISTS {SCHEMA_FQN}.stage_managed ("
#           f"  product_id INT, product_name STRING, price DOUBLE"
#           f") USING DELTA LOCATION '{MANAGED_SOURCE_PATH}'")
#   run_sql(f"INSERT OVERWRITE {SCHEMA_FQN}.stage_managed "
#           f"SELECT CAST(id AS INT) AS product_id, "
#           f"       concat('product_', id) AS product_name, "
#           f"       CAST(id * 1.5 AS DOUBLE) AS price "
#           f"FROM range(1, 101)")
#   run_sql(f"DROP TABLE {SCHEMA_FQN}.stage_managed")

# Repeat for external-source path.
# YOUR CODE: same pattern with stage_external + EXTERNAL_SOURCE_PATH.

# Now create the actual lab tables.

# products_managed: CREATE TABLE without LOCATION. UC copies the bytes
# into workspace storage and owns them.
# YOUR CODE:
#   run_sql(f"CREATE TABLE {MANAGED_TABLE_FQN} "
#           f"USING DELTA AS "
#           f"SELECT * FROM delta.`{MANAGED_SOURCE_PATH}`")
#   run_sql(f"ALTER TABLE {MANAGED_TABLE_FQN} SET TAGS "
#           f"('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

# products_external: CREATE TABLE with LOCATION. UC leaves the bytes
# in place and owns only the metastore registration.
# YOUR CODE:
#   run_sql(f"CREATE TABLE {EXTERNAL_TABLE_FQN} ("
#           f"  product_id INT, product_name STRING, price DOUBLE"
#           f") USING DELTA LOCATION '{EXTERNAL_SOURCE_PATH}'")
#   run_sql(f"ALTER TABLE {EXTERNAL_TABLE_FQN} SET TAGS "
#           f"('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

print(f"Managed table:  {MANAGED_TABLE_FQN}")
print(f"External table: {EXTERNAL_TABLE_FQN}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: Both tables exist with the right type and 100 rows. The
# external table's LOCATION matches EXTERNAL_SOURCE_PATH. The managed
# table's LOCATION is NOT under the student's bucket (UC copied the bytes
# into workspace-managed storage).


def _describe_extended(table_fqn):
    res = run_sql(f"DESCRIBE TABLE EXTENDED {table_fqn}", ignore_errors=True)
    info = {}
    for row in res.get("rows", []):
        if not row or len(row) < 2:
            continue
        key = (row[0] or "").strip()
        val = (row[1] or "").strip()
        if key:
            info[key] = val
    return info


def checkpoint_2(session):
    try:
        # Schema exists.
        catalog, parent, schema = SCHEMA_FQN.split(".")
        schema_sql = (
            "SELECT 1 FROM system.information_schema.schemata "
            f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
        )
        if not run_sql(schema_sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=f"Schema {SCHEMA_FQN} not found.",
            )

        # Both tables exist.
        for fqn in (MANAGED_TABLE_FQN, EXTERNAL_TABLE_FQN):
            cat, par, tbl = fqn.split(".")
            t_sql = (
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{cat}' AND table_schema = '{par}' "
                f"AND table_name = '{tbl}'"
            )
            if not run_sql(t_sql)["rows"]:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Table {fqn} not found.",
                )

        # Row counts.
        for fqn in (MANAGED_TABLE_FQN, EXTERNAL_TABLE_FQN):
            count = run_sql(f"SELECT COUNT(*) FROM {fqn}")["rows"][0][0]
            if int(count) != 100:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{fqn} has {count} rows; expected exactly 100."
                    ),
                )

        # Type check via DESCRIBE TABLE EXTENDED.
        managed_info = _describe_extended(MANAGED_TABLE_FQN)
        external_info = _describe_extended(EXTERNAL_TABLE_FQN)
        managed_type = managed_info.get("Type", "").upper()
        external_type = external_info.get("Type", "").upper()
        if "MANAGED" not in managed_type:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{MANAGED_TABLE_FQN} Type is {managed_type!r}; expected MANAGED. "
                    f"Did you accidentally include a LOCATION clause?"
                ),
            )
        if "EXTERNAL" not in external_type:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{EXTERNAL_TABLE_FQN} Type is {external_type!r}; expected EXTERNAL. "
                    f"Did you forget the LOCATION clause?"
                ),
            )

        # External table's LOCATION matches EXTERNAL_SOURCE_PATH.
        external_location = external_info.get("Location", "")
        normalized_external = (external_location or "").rstrip("/") + "/"
        if normalized_external != EXTERNAL_SOURCE_PATH:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{EXTERNAL_TABLE_FQN} Location is {external_location!r}; "
                    f"expected {EXTERNAL_SOURCE_PATH!r}."
                ),
            )

        # Managed table's LOCATION is NOT under the student's bucket.
        managed_location = managed_info.get("Location", "")
        if managed_location.startswith(EXTERNAL_LOCATION_URL.rstrip("/")):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{MANAGED_TABLE_FQN} Location is {managed_location!r}, which is "
                    f"under your bucket prefix. A managed table's location should be "
                    f"the workspace storage location (Databricks-managed bucket). Drop "
                    f"and recreate without a LOCATION clause."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is missing: schema, table existence, row count, table type, or location. Each lives in a separate `DESCRIBE TABLE EXTENDED` row or a separate count.

</details>

<details><summary>Hint 2 (direction)</summary>

Use a temporary external staging table at each source path, INSERT 100 rows via `range(1, 101)`, then DROP the staging table to leave only the bytes. The `products_managed` CREATE has NO `LOCATION` clause; the `products_external` CREATE HAS `LOCATION '<external-source>'`. Both must be `USING DELTA` so DESCRIBE TABLE EXTENDED reports the right Type.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
run_sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

# Stage at managed-source path.
run_sql(
    f"CREATE TABLE IF NOT EXISTS {SCHEMA_FQN}.stage_managed ("
    f"  product_id INT, product_name STRING, price DOUBLE"
    f") USING DELTA LOCATION '{MANAGED_SOURCE_PATH}'"
)
run_sql(
    f"INSERT OVERWRITE {SCHEMA_FQN}.stage_managed "
    f"SELECT CAST(id AS INT), concat('product_', id), CAST(id * 1.5 AS DOUBLE) "
    f"FROM range(1, 101)"
)
run_sql(f"DROP TABLE {SCHEMA_FQN}.stage_managed")

# Stage at external-source path.
run_sql(
    f"CREATE TABLE IF NOT EXISTS {SCHEMA_FQN}.stage_external ("
    f"  product_id INT, product_name STRING, price DOUBLE"
    f") USING DELTA LOCATION '{EXTERNAL_SOURCE_PATH}'"
)
run_sql(
    f"INSERT OVERWRITE {SCHEMA_FQN}.stage_external "
    f"SELECT CAST(id AS INT), concat('product_', id), CAST(id * 1.5 AS DOUBLE) "
    f"FROM range(1, 101)"
)
run_sql(f"DROP TABLE {SCHEMA_FQN}.stage_external")

# Managed table: no LOCATION clause -> UC copies bytes into workspace storage.
run_sql(
    f"CREATE TABLE {MANAGED_TABLE_FQN} USING DELTA AS "
    f"SELECT * FROM delta.`{MANAGED_SOURCE_PATH}`"
)
run_sql(f"ALTER TABLE {MANAGED_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

# External table: LOCATION clause -> UC leaves bytes in your bucket.
run_sql(
    f"CREATE TABLE {EXTERNAL_TABLE_FQN} ("
    f"  product_id INT, product_name STRING, price DOUBLE"
    f") USING DELTA LOCATION '{EXTERNAL_SOURCE_PATH}'"
)
run_sql(f"ALTER TABLE {EXTERNAL_TABLE_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
```

</details>

**Wallet check.** Still under $0.05. Two staging table creates, two inserts, two drops, plus the two real CREATE TABLEs. The serverless warehouse handled all of it in well under a minute.

## Task 3: Capture the managed table's location, DROP it, and prove the underlying files are gone

Before the DROP:

1. Run `DESCRIBE TABLE EXTENDED products_managed` and capture the `Location` value into `managed_location_pre_drop`. This is the workspace storage location (Databricks-managed bucket) where UC copied the bytes when you created the managed table without a LOCATION clause.
2. List the bucket prefix at that location to confirm Parquet/Delta files exist there now.

Then DROP the table and prove the bytes are gone:

3. `DROP TABLE products_managed`.
4. List the same prefix again. UC's managed-table contract is that the underlying files are removed when the metastore registration is dropped.

The validator queries `system.information_schema.tables` to confirm the table is unregistered, and compares the bucket LIST result before and after.

In [ ]:
# NBVAL_SKIP
# Task 3: Capture managed location, list bucket pre-drop, DROP TABLE, list
# bucket post-drop, prove the files are gone.

global managed_location_pre_drop

# YOUR CODE: Run DESCRIBE TABLE EXTENDED on MANAGED_TABLE_FQN and
# extract the Location string into managed_location_pre_drop.
# Example:
#   res = run_sql(f"DESCRIBE TABLE EXTENDED {MANAGED_TABLE_FQN}")
#   for row in res["rows"]:
#       if (row[0] or "").strip() == "Location":
#           managed_location_pre_drop = (row[1] or "").strip()
#           break

if not managed_location_pre_drop:
    print("managed_location_pre_drop is None. Run DESCRIBE TABLE EXTENDED first.")
    raise SystemExit(1)

# Pre-drop list. Should return at least one Parquet/Delta file.
pre_drop_list = list_path(managed_location_pre_drop)
print(f"Pre-drop list of {managed_location_pre_drop}:")
print(f"  files found: {len(pre_drop_list.get('rows', []))}")
print(f"  error:       {pre_drop_list.get('error')}")

# YOUR CODE: DROP TABLE MANAGED_TABLE_FQN.
#   run_sql(f"DROP TABLE {MANAGED_TABLE_FQN}")

# Give UC a moment to actually delete the files (typically immediate; some
# clouds add a tail second or two).
time.sleep(5)

# Post-drop list. Should return either no rows or a 'no such location' error.
post_drop_list = list_path(managed_location_pre_drop)
print(f"Post-drop list of {managed_location_pre_drop}:")
print(f"  files found: {len(post_drop_list.get('rows', []))}")
print(f"  error:       {post_drop_list.get('error')}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: managed_location_pre_drop is set; products_managed is no
# longer in information_schema.tables; the LIST against managed_location_pre_drop
# returns no Parquet/Delta files (the prefix may have been removed).


def checkpoint_3(session):
    try:
        if not managed_location_pre_drop:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "managed_location_pre_drop is empty. Capture the Location "
                    "from DESCRIBE TABLE EXTENDED before running DROP."
                ),
            )

        cat, par, tbl = MANAGED_TABLE_FQN.split(".")
        sql = (
            "SELECT 1 FROM system.information_schema.tables "
            f"WHERE table_catalog = '{cat}' AND table_schema = '{par}' "
            f"AND table_name = '{tbl}'"
        )
        if run_sql(sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{MANAGED_TABLE_FQN} is still registered in "
                    f"information_schema.tables. Run DROP TABLE before checking."
                ),
            )

        post_drop = list_path(managed_location_pre_drop)
        rows = post_drop.get("rows", []) or []
        # Filter to actual data files; ignore empty parent directory entry.
        data_files = [
            r for r in rows
            if isinstance(r, list) and len(r) > 1 and (
                ".parquet" in (r[1] or "").lower()
                or ".crc" in (r[1] or "").lower()
                or "_delta_log" in (r[1] or "").lower()
            )
        ]
        if data_files:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"After DROP TABLE on {MANAGED_TABLE_FQN}, the underlying "
                    f"location {managed_location_pre_drop!r} still has "
                    f"{len(data_files)} data files. UC's managed-table contract "
                    f"is that DROP removes the bytes. Confirm the table was "
                    f"actually managed (Type=MANAGED in DESCRIBE TABLE EXTENDED)."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint names what is missing: the captured location, the dropped registration, or residual files at the location. Each is one line of code or one DROP statement.

</details>

<details><summary>Hint 2 (direction)</summary>

`DESCRIBE TABLE EXTENDED <fqn>` returns rows of `(col_name, data_type, comment)`. The Location row has `col_name = "Location"`. Walk the rows until you find it, capture `row[1]`. Then `DROP TABLE <fqn>`. The post-drop `LIST` against the captured location should return no Parquet/Delta files.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
global managed_location_pre_drop

res = run_sql(f"DESCRIBE TABLE EXTENDED {MANAGED_TABLE_FQN}")
for row in res["rows"]:
    if (row[0] or "").strip() == "Location":
        managed_location_pre_drop = (row[1] or "").strip()
        break

run_sql(f"DROP TABLE {MANAGED_TABLE_FQN}")
```

If the post-drop LIST still returns Parquet files, the table was probably created with a LOCATION clause that landed under your bucket. Re-create the lab schema and run Task 2 again with the `CREATE TABLE products_managed USING DELTA AS SELECT ...` form (no LOCATION).

</details>

**Wallet check.** Still under $0.10. The managed-table drop did the file-delete work for free as part of the UC operation.

## Task 4: Capture the external table's pre-drop file count, DROP it, and prove the files survive

Before the DROP:

1. List `EXTERNAL_SOURCE_PATH` and capture the file count into `external_files_pre_drop`.

Then DROP and re-list:

2. `DROP TABLE products_external`.
3. List `EXTERNAL_SOURCE_PATH` again. The file count must equal `external_files_pre_drop` (the bytes are intact).

This is the headline lesson of the lab: external tables only own metastore registration. The bytes belong to the external location; DROP TABLE removes the registration but leaves the files. If you want to clean up the bytes after dropping an external table, you must explicitly delete them at the bucket layer (the cleanup cell does this in `aws s3 rm` / `az storage blob delete-batch` / `gsutil rm`).

In [ ]:
# NBVAL_SKIP
# Task 4: Capture pre-drop file count at EXTERNAL_SOURCE_PATH, DROP TABLE,
# re-list, prove files are intact.

global external_files_pre_drop

# YOUR CODE: List EXTERNAL_SOURCE_PATH and assign the count of returned
# rows to external_files_pre_drop.
# pre_list = list_path(EXTERNAL_SOURCE_PATH)
# external_files_pre_drop = len(pre_list.get("rows", []))

if external_files_pre_drop is None:
    print("external_files_pre_drop is None. List the path first and assign the count.")
    raise SystemExit(1)

print(f"Pre-drop file count at {EXTERNAL_SOURCE_PATH}: {external_files_pre_drop}")

# YOUR CODE: DROP TABLE EXTERNAL_TABLE_FQN.
#   run_sql(f"DROP TABLE {EXTERNAL_TABLE_FQN}")

# Give the metastore a moment to settle the metadata change.
time.sleep(3)

post_list = list_path(EXTERNAL_SOURCE_PATH)
post_count = len(post_list.get("rows", []))
print(f"Post-drop file count at {EXTERNAL_SOURCE_PATH}: {post_count}")
print()
print("The files survived the drop. They belong to the external location, not")
print("to Unity Catalog. The cleanup cell at the bottom of this notebook deletes")
print("them explicitly via the bucket-prefix cleanup adapter; if you stop here,")
print("the bytes will accrue cloud-side storage charges.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: external_files_pre_drop is set, products_external is no
# longer registered, and the post-drop LIST has the same file count as
# pre-drop (the bytes survived).


def checkpoint_4(session):
    try:
        if external_files_pre_drop is None:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "external_files_pre_drop is None. Capture the pre-drop file "
                    "count via list_path() before DROP TABLE."
                ),
            )
        if int(external_files_pre_drop) <= 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"external_files_pre_drop is {external_files_pre_drop}; expected "
                    f">0. Confirm Task 2 actually wrote bytes to {EXTERNAL_SOURCE_PATH}."
                ),
            )

        cat, par, tbl = EXTERNAL_TABLE_FQN.split(".")
        sql = (
            "SELECT 1 FROM system.information_schema.tables "
            f"WHERE table_catalog = '{cat}' AND table_schema = '{par}' "
            f"AND table_name = '{tbl}'"
        )
        if run_sql(sql)["rows"]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"{EXTERNAL_TABLE_FQN} is still registered. Run DROP TABLE."
                ),
            )

        post_list = list_path(EXTERNAL_SOURCE_PATH)
        post_count = len(post_list.get("rows", []) or [])
        if post_count != int(external_files_pre_drop):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Post-drop file count at {EXTERNAL_SOURCE_PATH} is {post_count}; "
                    f"expected {external_files_pre_drop} (the same as pre-drop). The "
                    f"external table's bytes should have survived the DROP."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=str(exc))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Two lines: capture the pre-drop count via `len(list_path(EXTERNAL_SOURCE_PATH)["rows"])`, then DROP. The post-drop count should match.

</details>

<details><summary>Hint 2 (direction)</summary>

`list_path()` (defined in the setup cell) wraps `LIST '<uri>'` and returns a dict with `rows`. Each row is `[path, name, size, modificationTime]`. The simplest pre-drop count is `len(pre_list.get("rows", []))`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

```python
global external_files_pre_drop

pre_list = list_path(EXTERNAL_SOURCE_PATH)
external_files_pre_drop = len(pre_list.get("rows", []))

run_sql(f"DROP TABLE {EXTERNAL_TABLE_FQN}")
```

If the post-drop count is zero, the table was probably MANAGED, not EXTERNAL (UC removed the bytes on drop). Re-check Checkpoint 2's `Type=EXTERNAL` requirement.

</details>

**Wallet check.** Still under $0.20 for the session. Two DROP TABLEs and a few LIST calls. The bigger line item, if you skip cleanup, is the orphaned bytes at `<bucket-prefix>external-source/` that will keep accruing storage charges in your cloud account.

## Cleanup

The cleanup cell drops the external location, the storage credential, the schema (cascades the remaining tables if any survived), and prints a reminder for the cloud-side principal and the bucket-side bytes that the notebook cannot reach.

What this notebook can clean up automatically:

- The Unity Catalog objects (storage credential, external location, schema, any remaining tables).

What you must clean up manually:

- The bytes at `<bucket-prefix>external-source/` (and `<bucket-prefix>managed-source/` if Task 2's staging tables left anything). Use:
  - AWS: `aws s3 rm '<bucket-prefix>' --recursive`
  - Azure: `az storage blob delete-batch --source <container> --pattern 'labrun-external/*'`
  - GCP: `gsutil rm -r '<bucket-prefix>'`
- The cloud-side IAM role / managed identity / service account.
- The Premium trial workspace itself before day 14 of the trial. Set a calendar reminder NOW.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every UC object in CLEANUP_MANIFEST. Print explicit
# reminders for cloud-side cleanup the notebook cannot perform.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and getattr(r, "critical", False)]
standard_resources = [r for r in CLEANUP_MANIFEST if r.id not in failed_ids and not getattr(r, "critical", False)]
critical_destroyed = len(critical_resources)
standard_destroyed = len(standard_resources)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")
print()
print("MANUAL CLEANUP REMINDERS (notebook cannot reach these):")
print(f"  1. Delete bucket bytes:")
print(f"     aws s3 rm '{EXTERNAL_LOCATION_URL}' --recursive")
print(f"     (or the Azure / GCP equivalent for your cloud)")
print(f"  2. Delete the cloud-side principal you created in Task 1.")
print(f"  3. Tear down the Premium trial workspace before the 14-day trial expires,")
print(f"     or it converts to paid and starts billing your cloud account.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $0.05 to $0.50 on your cloud account.** The Premium trial itself stayed at $0 inside the 14-day window, but the underlying serverless compute and the bucket Parquet writes ran against your cloud quota. The cleanup cell tells you exactly what bytes still live in your bucket and what date the trial converts; act on both before you close the notebook.

## Reflection

These are not graded. They are for you.

1. Walk through what Unity Catalog does at `CREATE TABLE` time when you specify `LOCATION '<path>'` vs when you omit `LOCATION`. Name each step: the storage-credential lookup, the external-location resolution, the file ownership decision, the metastore registration. Why does `DROP TABLE` on an external table NOT delete the files?

2. Your migration team has two datasets to register in Unity Catalog: (a) a master customer dataset that security insists must stay in their existing S3 bucket, (b) an analyst's ephemeral feature-engineering scratch table. Which gets managed and which gets external, and why? What would happen if you reversed the choice?

## Exam-Style Practice

**Question 1 (Domain 7, managed vs external semantics):**

A data engineer runs `DROP TABLE catalog.schema.products_external` where `products_external` is an external Delta table whose LOCATION points at `s3://team-bucket/products/`. After the drop, what is the state of the table and the bucket?

A. The table registration is removed from Unity Catalog; the Parquet files at `s3://team-bucket/products/` are also removed because UC owns the path.
B. The table registration is removed from Unity Catalog; the Parquet files at `s3://team-bucket/products/` remain intact because external tables do not own the underlying files.
C. The drop fails with `EXTERNAL_TABLE_DROP_NOT_ALLOWED`; external tables can only be dropped after first being converted to managed.
D. The table registration AND the files are removed, but the `_delta_log` directory at `s3://team-bucket/products/_delta_log/` is preserved for forensic recovery.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: UC does NOT own the files of an external table; the storage credential and external location define a trust relationship but the files belong to the external location, not to UC.
- B is correct: DROP TABLE on an external Delta table removes the metastore registration only. The Parquet files and the `_delta_log` directory remain in place. If the engineer wants to clean up the bytes, they must explicitly delete the cloud-bucket prefix.
- C is wrong: DROP TABLE on an external table is fully supported; no conversion is required.
- D is wrong: nothing about Delta or UC preserves `_delta_log` after a file-deleting operation. In this scenario, no file deletion happens at all.

</details>

**Question 2 (Domain 7, storage credential and external location):**

A team wants to register an existing S3 bucket of Parquet files as an external Delta table in Unity Catalog. The bucket is in the team's own AWS account; Databricks is in a separate account. Which sequence is correct?

A. Create the external table directly with `CREATE TABLE ... LOCATION 's3://team-bucket/data/'`; Databricks will infer the cross-account access automatically.
B. Create a storage credential bound to an IAM role in the team's AWS account, create an external location over the bucket prefix using that credential, then create the external table with `LOCATION '<path-under-external-location>'`.
C. Make the bucket public, then create the external table; private buckets cannot be registered.
D. Copy the data into the Databricks-managed bucket first, then register a managed table; external tables are not supported for cross-account data.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: Databricks cannot infer cross-account access. A storage credential is required to broker the trust.
- B is correct: the documented Unity Catalog pattern is storage credential (cross-account IAM role trust) plus external location (UC-side governance of the bucket prefix) plus CREATE TABLE with LOCATION under the external location. Each layer is required.
- C is wrong: public buckets are an anti-pattern and not required for cross-account access.
- D is wrong: external tables fully support cross-account data; the entire point of the storage credential plus external location pattern is to enable it without copying data.

</details>